#### 농림축산검역본부 철새 이동 정보 API 수집 ➡️ Bronze 테이블 적재
##### Blob Storage raw CSV → Delta Bronze 적재
##### Blob 경로: raw/bird/batch={batch_id}/{recv_id}.json

In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG, BASE_PATH
from datetime import date

# 이번 분기 bird 파일 없으면 스킵
quarter = f"{date.today().year}Q{(date.today().month - 1) // 3 + 1}"
try:
    dbutils.fs.ls(f"{BASE_PATH}/bird/quarter={quarter}/")
except Exception:
    dbutils.notebook.exit("SKIPPED")

# 조류 관측 데이터 적재 (batch 파티션, 스키마 병합)
df_bird = spark.read.option("mergeSchema", "true").json(f"{BASE_PATH}/bird/batch=1/")
df_bird.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.raw.bird")

print(df_bird.count())